# Variant 3 — TaskTransformer (task) + RegionTransformer + TimeTransformer (separated)
Pipeline cascata: predice prima il task, poi la regione condizionata al task, poi il tempo.

In [ ]:
import os
import torch
import pandas as pd
from config import DATA_DIR

from pm4py.algo.conformance.alignments.petri_net import algorithm as alignments
from pm4py.objects.log.obj import Trace, Event, EventLog

from core import TaskTransformer, RegionTransformer, TimeTransformer
from core.training import train_task, train_region, train_time_v2
from utils import get_decoding, get_encoding, hamming_distance, edit_distance_weighted_levenshtein

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
DATA_FILE = DATA_DIR / 'prepared_data_sepsis.pt'

In [ ]:
info = torch.load(DATA_FILE, map_location=device, weights_only=False)
n = info['n']

data_task = {
    'train_tasks': info['data_tasks'][:n],
    'val_tasks': info['data_tasks'][n:],
}
data_region = {
    'train_tasks': info['data_tasks'][:n],
    'val_tasks': info['data_tasks'][n:],
    'train_regions': info['data_regions'][:n],
    'val_regions': info['data_regions'][n:],
}
data_time = {
    'train_tasks': info['data_tasks'][:n],
    'val_tasks': info['data_tasks'][n:],
    'train_regions': info['data_regions'][:n],
    'val_regions': info['data_regions'][n:],
    'train_times': info['data_times'][:n],
    'val_times': info['data_times'][n:],
}

vocab_size_tasks = info['vocab_size_tasks']
vocab_size_regions = info['vocab_size_regions']
num_regions = info['num_regions']
num_tasks = info['num_tasks']

decode_tasks = lambda b: [info['id_to_bit_tasks'][x] for x in b]
decode_regions = lambda b: [info['id_to_bit_regions'][x] for x in b]
encode_tasks = lambda a: [info['bit_to_id_tasks'][tuple(x)] for x in a]
encode_regions = lambda a: [info['bit_to_id_regions'][tuple(x)] for x in a]

net = info['net']

print(f'vocab_tasks={vocab_size_tasks}, vocab_regions={vocab_size_regions}, n_train={n}')

In [ ]:
param_task_transformer_default = dict(block_size=256, n_embd=256, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=2500, eval_iters=200, eval_interval=100, patience=4, use_swa=True, swa_start_ratio=0.6, diverge_threshold=1.5)
param_region_transformer_default = dict(block_size=256, n_embd=256, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=2500, eval_iters=200, eval_interval=100, patience=4, use_swa=True, swa_start_ratio=0.6, diverge_threshold=1.5)
param_time_transformer_default = dict(block_size=64, n_embd=128, n_head=8, n_layer=3, dropout=0.30, lr=3e-4, weight_decay=0.01, batch_size=16, max_iters=2500, eval_iters=200, eval_interval=100, patience=4, use_swa=True, swa_start_ratio=0.6, diverge_threshold=1.5)

params = torch.load(DATA_DIR / 'v3_best_params.pt', map_location=device, weights_only=False) if os.path.exists(DATA_DIR / 'v3_best_params.pt') else None

p_task = param_task_transformer_default
p_region = param_region_transformer_default
p_time = param_time_transformer_default
if params is not None:
    p_task = {**params['TaskTransformer'], 'max_iters': 10000, 'eval_iters': 100, 'eval_interval': 100, 'patience': 6, 'use_swa': True, 'swa_start_ratio': 0.6, 'diverge_threshold': 1.5}
    p_region = {**params['RegionTransformer'], 'max_iters': 10000, 'eval_iters': 100, 'eval_interval': 100, 'patience': 6, 'use_swa': True, 'swa_start_ratio': 0.6, 'diverge_threshold': 1.5}
    p_time = {**params['TimeTransformer'], 'max_iters': 10000, 'eval_iters': 100, 'eval_interval': 100, 'patience': 6, 'use_swa': True, 'swa_start_ratio': 0.6, 'diverge_threshold': 1.5}

print('Task:', p_task)
print('Region:', p_region)
print('Time:', p_time)

In [ ]:
model_task = TaskTransformer(
    task_vocab_size=vocab_size_tasks,
    block_size=p_task['block_size'],
    n_embd=p_task['n_embd'],
    dropout=p_task['dropout'],
    n_head=p_task['n_head'],
    n_layer=p_task['n_layer'],
).to(device)

train_task(model_task, data_task, p_task, device, data_key='tasks', printing=True)
print('TaskTransformer trained.')

In [ ]:
model_region = RegionTransformer(
    region_vocab_size=vocab_size_regions,
    task_vocab_size=vocab_size_tasks,
    block_size=p_region['block_size'],
    n_embd=p_region['n_embd'],
    dropout=p_region['dropout'],
    n_head=p_region['n_head'],
    n_layer=p_region['n_layer'],
).to(device)

train_region(model_region, data_region, p_region, device, printing=True)
print('RegionTransformer trained.')

In [ ]:
model_time = TimeTransformer(
    vocab_size_task=vocab_size_tasks,
    block_size=p_time['block_size'],
    n_embd=p_time['n_embd'],
    dropout=p_time['dropout'],
    n_head=p_time['n_head'],
    n_layer=p_time['n_layer'],
    vocab_size_region=vocab_size_regions,
    separated_regions=True,
).to(device)

train_time_v2(model_time, data_time, p_time, device, printing=True)
print('TimeTransformer (v2) trained.')

In [ ]:
max_new_tokens = 100

sep_task_id = info['bit_to_id_tasks'][tuple([0]*num_tasks)]
sep_region_id = info['bit_to_id_regions'][tuple([0]*num_regions)]
sep_mask = (info['data_tasks'][:n] == sep_task_id) & (info['data_regions'][:n] == sep_region_id)
mean_sep_delta = info['data_times'][:n][sep_mask].float().mean().item()

context_task = torch.tensor(encode_tasks([[0]*num_tasks]), dtype=torch.long, device=device).unsqueeze(0)
context_region = torch.tensor(encode_regions([[0]*num_regions]), dtype=torch.long, device=device).unsqueeze(0)
context_time = torch.tensor([mean_sep_delta], dtype=torch.float32, device=device).unsqueeze(0) # Lo zero sembra corretto al momento (da verificare, sembrerebbe andare bene anche 1)

gen_task_ids, gen_region_ids, gen_times = [], [], []

for _ in range(max_new_tokens):
    next_task = model_task.predict_next_task(idx_task=context_task, block_size=p_task['block_size'])
    context_task = torch.cat((context_task, next_task), dim=1)
    context_task_aligned = context_task[:, 1:]   # allineamento offset

    next_region = model_region.predict_next_region(idx_region=context_region, idx_task=context_task_aligned, block_size=p_region['block_size'])
    context_region = torch.cat((context_region, next_region), dim=1)
    context_region_aligned = context_region[:, 1:]

    next_time = model_time.predict_next_time(idx_tasks=context_task_aligned, idx_regions=context_region_aligned, idx_times=context_time, block_size=p_time['block_size'])
    context_time = torch.cat((context_time, next_time), dim=1)

    gen_task_ids.append(next_task.item())
    gen_region_ids.append(next_region.item())
    gen_times.append(next_time.item())

decoded = []
for i in range(max_new_tokens):
    t_bits = [int(b) for b in decode_tasks([gen_task_ids[i]])[0]]
    r_bits = [int(b) for b in decode_regions([gen_region_ids[i]])[0]]
    combined = r_bits + t_bits
    decoded.append(combined)
    #print(f'{i:02d}: {combined} — time={round(gen_times[i],4)}')

In [ ]:
# Raggruppa la sequenza in tracce separate (separatore = vettore zero)
traces_generated, current = [], []
for step in decoded:
    current.append(step)
    if step == [0]*(num_regions+num_tasks):
        if len(current) > 1:
            traces_generated.append(current)
        current = []

for i,trace in enumerate(traces_generated):
    print(f"{i}: {trace}")

In [ ]:
traces_decoded = get_decoding(traces_generated, net.regions, net.tasks)
print(traces_decoded)

In [ ]:
'''
PROBLEMA: Può generare tracce sfasate, con più eventi per step.
get_decoding non funziona sotto questo punto di vista. o meglio, funziona generando eventi in più (tipo 6 eventi da 4 step perchè ci sono 2 step generati male)
'''

classifier_dict_tasks = info['classifier_dict_tasks']
dict_task_step_encoding = info['dict_task_step_encoding']
current_trace_context = []
#traces_decoded_list = [step for trace in traces_decoded for step in trace]

tasks_previous = [0] * num_tasks
for i, (step, t) in enumerate(zip(decoded, gen_times)):
    bits = [int(b) for b in step]
    is_sep = bits == [0]*(num_regions+num_tasks)
    tasks_step = bits[num_regions:]

    step_events = []
    for j, task in enumerate(tasks_step):
        if task != tasks_previous[j]:
            step_events.append(("start_" if task == 1 else "end_") + net.tasks[j])
    tasks_previous = tasks_step

    if len(step_events) == 1:
        event_name = step_events[0]
        if event_name in classifier_dict_tasks:
            rt, max_len = classifier_dict_tasks[event_name]
            ctx = list(reversed(current_trace_context))[:max_len]
            padded = ctx + ['PAD'] * (max_len - len(ctx))
            encoded = [dict_task_step_encoding.get(s, dict_task_step_encoding['PAD']) for s in padded]
            expected = round(rt.predict([encoded])[0], 4)
        else: # Non ci dovrebbe mai entrare in teoria
            expected = 0.0 if not current_trace_context else "n/d"
        current_trace_context.append(event_name)
        note = ""
    elif len(step_events) == 0: # step che non genera eventi (es. cambia solo la regione)
        expected, note = "—", "(step senza evento)"
    else:# step malformato: accende/spegne 2 task insieme
        expected, note = "—", f"(step ambiguo: {step_events})"

    if is_sep:
        current_trace_context = []

    print(f'{i:02d}: {bits} — time={round(t,4)} | expected={expected} {note}')

In [ ]:
# Allineamento con conformance checking pm4py
check = ["P", "X", "L"]
start = tuple(["start_" + c for c in check])
end = tuple(["end_" + c for c in check if c!="L"]) # si escludono gli end loop
loop = tuple(["back_L"]) + tuple(["end_L"])
silent_prefixes = start + end + loop

# Creo i parametri per l'allineamento
model_cost, sync_cost = {}, {}
for t in net.net.transitions: # Prendo tutte le transizioni
    if t.label is None or (t.label is not None and t.label.startswith(silent_prefixes)): # Se è una transizione silente
        model_cost[t] = 0
        sync_cost[t] = 10000
    else: # Se è un task vero e proprio
        model_cost[t] = 10000
        sync_cost[t] = 0

alignment_params = {
    alignments.Parameters.PARAM_MODEL_COST_FUNCTION: model_cost,
    alignments.Parameters.PARAM_SYNC_COST_FUNCTION: sync_cost,
}

# Creiamo l'EventLog di ogni traccia per poi poterla allineare
eventlog_traces = EventLog()
for trace in traces_decoded:
    t = Trace()
    for activity in trace:
        t.append(Event({'concept:name': activity}))
    eventlog_traces.append(t)

aligned_traces = alignments.apply(eventlog_traces, net.net, net.initial_marking, net.final_marking, parameters=alignment_params)

for i,trace in enumerate(aligned_traces):
    print(f"{i}: {trace}")

In [ ]:
'''Codifichiamo le tracce allineate (per poi poterle confrontare con quelle generate dal transformer)'''

silent_prefixes = start + end + tuple(["back_L"])

aligned_traceEncoded_regions, aligned_traceEncoded_tasks = get_encoding(
    [[step for _, step in a['alignment'] if step and not step.startswith(silent_prefixes) and step != '>>']
     for a in aligned_traces],
    net.regions, net.tasks, net.open_clauses, net.end_clauses
)

print(aligned_traceEncoded_regions)

df_aligned_traces = pd.concat([aligned_traceEncoded_regions, aligned_traceEncoded_tasks], axis=0)

df_aligned_traces

In [ ]:
'''Creo una lista delle tracce codificate (ogni traccia è una lista dove ogni elemento è una colonna del df, ossia uno step) --> più facili da confrontare quando calcoliamo la distanza'''

aligned_traces_encoded = []
aligned_trace_encoded = []
for element in df_aligned_traces.T.values:
    element = element.tolist()
    aligned_trace_encoded.append(element)
    if element == [0] * (num_regions+num_tasks):
        aligned_traces_encoded.append(aligned_trace_encoded)
        aligned_trace_encoded = []

# Andiamo a calcolare il costo con la edit distance (weighted_levenshtein)
costs = []
for i, (gen, aln) in enumerate(zip(traces_generated, aligned_traces_encoded)):
    cost = edit_distance_weighted_levenshtein(gen, aln, num_regions+num_tasks, num_regions+num_tasks, hamming_distance)
    costs.append(cost)
    print(f'Traccia {i}: edit_distance={cost}')

print(f'\nEdit distance media: {sum(costs)/len(costs):.2f}')